In [10]:
from nimber_ops import *
from transfinite import Ordinal

ImportError: cannot import name 'w' from 'transfinite' (unknown location)

In [25]:
# utility functions: collect into util.py later
def base(n : int, b : int, length :  int = 0) -> list[int]:
    ''' returns the base b expansion of n as a  list
    n = sum_j^N a_j*b^j returns [a_0, a_1, ..., a_N]
    optional length parameter to include leading zeros if desired,
    otherwise minimum possible length where highest digit is non-zero'''
    assert (n >= 0 and b > 1), ('positional base expansion defined for' 
                                'non-negative integers and positive bases')
    if length == 0:
        if n == 0: return [0]
        coeffs = []
        while n > 0:
            coeffs.append(n % b)
            n //= b
        return coeffs
    else:
        coeffs = base(n, b)
        while len(coeffs) < length:
            coeffs.append(0)
        return coeffs

def base_eval(coeffs : list[int], b : int) -> int:
    total = 0
    for coeff in coeffs[::-1]:
        total = coeff + b * total
    return total
  
def ord_decomp(ordinal : Ordinal | int) -> list:
    ''' returns [inf_1, inf_2, ..., inf_n, finite term]'''
    if isinstance(ordinal, int):
        assert ordinal >= 0
        return [ordinal]
    high = Ordinal(ordinal.exponent, ordinal.coefficient)
    terms = [high]
    remainder = ordinal.addend
    while isinstance(remainder, Ordinal):
        terms.append(Ordinal(remainder.exponent, remainder.coefficient, 0))
        remainder = remainder.addend
    assert(isinstance(remainder, int))
    terms.append(remainder)
    return terms

def ord_recomp(list : list) -> Ordinal | int:
    result = 0
    terms = sorted(list, reverse=True)
    for term in list:
        result = result + term
    return result

small_primes = [2,3,5,7,11,13,17,19,23,29,31,37,41,43,47,53,59,61,67]
kappa_p = { # generator of smallest extension of prime degree p
           # kappa_{p_^n} = w^{w^{k-1}*p^{n-1}} where k = |{primes < p}|
           2: 2, 3 : Ordinal()}
for i, prime in enumerate(small_primes[2:]):
    kappa_p[prime] = Ordinal(Ordinal(i+1))
alpha_p = { # (kappa_p)^p, e.g. w^3=2, [w^w]^5 = 4, [w^w^2]^7 = w+1, etc.
           # very hard to calculate for higher p, based on http://www.neverendingbooks.org/on2-extending-lenstras-list/
            2:3, 3:2, 5:4, 7:Ordinal()+1, 11:Ordinal(Ordinal())+1, 13:Ordinal()+4,
            17: 16, 19:Ordinal(3)+4, 23:Ordinal(Ordinal(3))+1, 
            29:Ordinal(Ordinal(2))+4, 31:Ordinal(Ordinal())+1, 37: Ordinal(3)+4,
            41: Ordinal(Ordinal())+1, 43:Ordinal(Ordinal(2))+1, 47:Ordinal(Ordinal(7))+1, 
            53: Ordinal(Ordinal(4))+1, 59:Ordinal(Ordinal(8))+1, 
            61:Ordinal(Ordinal())+Ordinal(), 67:Ordinal(Ordinal(3))+Ordinal()}

In [225]:
kappa_p[67]

w**w**17

In [ ]:
class Nim:
    ''' nimbers '''
    def __init__(self, n : int | Ordinal) -> None:
        ''' ordinal considered an a field element in On_2 
        val = ordinal
        field = smallest x > n such that x is a field
        base = largest y < n such that y is a field (or base = 0 for n < 2)
        write n = high * base + low, where low,high < base
        '''
        def exp2(level : int) -> int:
            return 1 << level
        if isinstance(n, int):
            self.val = abs(n)
            self.isfinite = True            
            if self.val < 2:
                self.field = 2 # smallest 
                self.base = 0
                self.high = 0
                self.low = self.val
            else:
                level = 0
                while n >> (1 << level) > 0:
                    level += 1
                self.field = exp2(exp2(level))
                self.base = exp2(exp2(level - 1))
                self.high = self.val // self.base
                self.low = self.val - self.high * self.base
        else:
            assert isinstance(n, Ordinal), 'An infinite nimber must be an ordinal'
            self.val = n
            self.isfinite = False
            # find the smallest field containing n
            if (k:=n.exponent) < Ordinal(): # n = omega^k, k fintie => cubic extension
                power = 0
                while 3 ** power <= k: 
                    power += 1
                self.field = Ordinal(3**power)
            exp = 3**(power-1)
            self.base = Ordinal(exp)
            high = Ordinal(n.exponent - exp, n.coefficient, 0) if exp < n.exponent else n.coefficient
            remainder = n.addend
            while isinstance(remainder, Ordinal) and remainder.exponent > exp:
                high += Ordinal(remainder.exponent - exp, remainder.coefficient, 0)
                remainder = remainder.addend
                if isinstance(remainder, Ordinal) and remainder.exponent == exp: 
                    high += remainder.coefficient
                    remainder = remainder.addend
            self.high = high
            self.low = remainder

    def __add__(self, other):
        if self.isfinite and other.isfinite:
            return Nim(self.val ^ other.val) # finite nim sum is bitwise XOR
        if self.val == other.val: 
            return Nim(0)
        ord1, ord2 = self.val, other.val
        terms1, terms2 = ord_decomp(ord1), ord_decomp(ord2)
        sum = {} 
        for term in terms1[:-1]:
            sum[term.exponent] = term.coefficient
        for term in terms2[:-1]: # nim sum the coeffs if any terms with same exp
            try: sum[term.exponent] = sum[term.exponent] ^ term.coefficient
            except: sum[term.exponent] = term.coefficient
        
        keys = sorted(sum.keys()) #  ordinal addition not commutative
        result = terms1[-1] ^ terms2[-1] # nim sum of finite part
        for key in keys:
            if sum[key] > 0:
                result = Ordinal(key, sum[key]) + result # add bigger on the left
        return Nim(result)
            
    def __eq__(self, other):
        return self.val == other.val

    def __hash__(self):
        return self.val.__hash__()
    
    def __mul__(self, other):
        assert(isinstance(other, Nim))
        x, y = self.val, other.val
        if x == 0 or y == 0: return Nim(0)
        if x == 1: return other
        if y == 1: return self
        if self.isfinite and other.isfinite:
            def nim_product(a : int, b : int) -> int:
                # first handle trivial cases
                if a == 0 or b == 0:
                    return 0
                elif a == 1:
                    return b
                elif b == 1:
                    return a
                elif a == 2 and b == 2:
                    return 3
                else:
                    # do euclidean division by greatest possible fermat power 
                    # a = q_a * F_a + r_a and b = q_b * F_b + r_b
                    F_a, q_a, r_a = Nim(a).base, Nim(a).high, Nim(a).low
                    F_b, q_b, r_b = Nim(b).base, Nim(b).high, Nim(b).low
                    
                    # if one the Fermat powers is greater than the other, then
                    # nim multiplication by it is the same as ordinary multiplication
                    if F_a < F_b:
                        return nim_product(a,q_b)*F_b ^ nim_product(a,r_b)
                    elif F_a > F_b:
                        return nim_product(q_a,b)*F_a ^ nim_product(r_a,b)
                    else:
                        # otherwise we have to distribute and use F_n ** 2 = 3 * F_n / 2
                        p_1 = nim_product(q_a,q_b)
                        p_2 = nim_product(r_a,r_b)
                        p_3 = nim_product(q_a ^ r_a, q_b ^ r_b)
                        p_4 = nim_product(p_1, F_a >> 1)
                        p_5 = p_3 ^ p_2
                        return p_5 * F_a ^ p_2 ^ p_4
            return Nim(nim_product(x, y))  
        elif self.isfinite and not other.isfinite:
            terms = ord_decomp(other.val) # distribute to each coefficient
            terms[-1] = (self * Nim(terms[-1])).val
            for term in terms[:-1]:
                term.coefficient = (self * Nim(term.coefficient)).val
            return Nim(ord_recomp(terms))
        elif not self.isfinite and other.isfinite:
            return other * self
        else: # both infinite
            terms1, terms2 = ord_decomp(self.val), ord_decomp(other.val)
            inf1, fin1 = terms1[:-1], terms1[-1]
            inf2, fin2 = terms2[:-1], terms2[-1]
            # start by "FOIL-ing" to handle the terms where one is finite
            result = Nim(fin1) * other + self * Nim(fin2) + Nim(fin1) *Nim(fin2)
            for x in inf1: # expand and distribute the purely infinite terms
                for y in inf2: 
                    # calculate (w^N * a) x (w^M * b)
                    N, M = x.exponent, y.exponent
                    a, b = x.coefficient, y.coefficient
                    # currently only implementing < w^w
                    # want to eventually go up to < w^w^18 using alpha_p table
                    assert N < Ordinal() and M < Ordinal()
                    N_tern = base(N, 3) # write exponents in ternary
                    M_tern = base(M, 3)
                    K = max([len(N_tern), len(M_tern)])
                    coeff = Nim(a) * Nim(b)
                    
                    # invert the powers of 3 since w^(3^k) = 2^(3^{-k-1})
                    phi = lambda n: base_eval(base(n, 3, K)[::-1], 3) #reversed 3s
                    exp_2 = phi(N) + phi(M) # the exponent 2^(3^{-K} * exp_2)

                    next_power = 3 ** K
                    q, r = exp_2 // next_power, exp_2 % next_power
                    coeff = coeff * Nim(2) ** q
                    W = Nim(Ordinal(phi(r))) if r > 0 else Nim(1)
                    result = result + coeff * W
            return result
    def __pow__(self, p):
        """
        Compute x**n using exponentiation by squaring.

        """
        if p >= 0: # binary exponentiation by squaring
            result = Nim(1) 
            nimber = Nim(self.val)
            while p > 0:
                if p & 1:
                    result = result * nimber
                nimber = nimber * nimber
                p >>= 1
            return result
        elif p == -1:
            if self.field == 2: return self
            a, b, F, f = Nim(self.high), Nim(self.low), Nim(self.base), Nim(self.base >> 1)
            det = (a + b)*b + a*a*f
            return det**(-1) * (a*F + (a+b))
        else:
            inv = self ** (-1)
            return inv ** (-p)       
            
    def __repr__(self) -> str:
        if self.isfinite:       
            return str(self.val)
        else:
            return self.val.__repr__()  
    
    def _repr_latex_(self):
        """
        Special method for Jupyter to render LaTeX.
        """
        if self.isfinite:
            # No special LaTeX for integers, just return the string
            return f"${self.val}$"
        else:
            # Delegate to the Ordinal's LaTeX representation
            return self.val._repr_latex_()
        
    def inv(self):
        return self ** (-1)           
 
    def order(self):
        if self.isfinite:
            def fermat_divisors(n : int, include_one : bool = False ) -> list:
                '''
                Find the divisors of a Mersenne number 2 ** (2 ** n) - 1
                By default does not include 1
                '''
                # for now, this only works for n < 6
                # could potentiall go up to n = 11 using known factors on wikipedia 
                # no one knows the factors of 2 ** (2 ** 11) + 1
                if n >= 6:
                    raise ValueError('This function only works for n < 6')
                else:
                    divisors = []
                    for i in range(0 + int(not include_one),2 ** n):
                        product = 1
                        for j in range(n):
                            if i >> j & 1:
                                product *= 2 ** (2 ** j) + 1
                        divisors.append(product)
                    return divisors
                
            n = self.val
            if n == 0:
                return 0
            elif n == 1:
                return 1
            elif n in {2, 3}:
                return 3
            elif n == 3:
                return 3
            elif n < 1 << (1 << 5):
                # make more efficient by only checking possible orders
                # use Lagrange's theorem
                # find the smallest field containing n i.e. smallest F_k > n
                exp = (n.bit_length() - 1).bit_length()
                # find the order of n must divide F_k - 1 which factors by difference of squares
                divisors = fermat_divisors(exp)
                for factor in divisors[:-1]:
                    if (Nim(n)**factor).val == 1:
                        return factor
                else:
                    return divisors[-1] 
            else:
                for factor in fermat_divisors(5):
                    if (Nim(n)**factor).val == 1:
                        return factor
                    # brute force: will probably loop forever
                    i = 1 << (1 << 5) + 1
                    while True:
                        if (Nim(i)**factor).val == 1:
                            return i
                        i += 2
        else:
            ... # inifite case is hard..
                    
    def sqrt(self):
        if self.isfinite:
            if self.field == 2:
                return self
            term = self**2 + self
            return term.sqrt() + self
        else:
            ... # not sure how to implement...
 

NameError: name 'Ordinal' is not defined

In [35]:
for n in range(5):
    F = 1<<(1<<n)
    x = Nim(F)
    ord_x = x.order()
    quot = (F**2 - 1)//ord_x
    print(f'order of {x} is {ord_x}, which is {quot}^th root of gen')
    n=0
    while ord_x < F**2-1:
        x = Nim(x.val + 1)
        ord_x = x.order()
        n+=1
    print(f'smallest generator of {F**2} is {x}, which is {n} more')

order of 2 is 3, which is 1^th root of gen
smallest generator of 4 is 2, which is 0 more
order of 4 is 15, which is 1^th root of gen
smallest generator of 16 is 4, which is 0 more
order of 16 is 85, which is 3^th root of gen
smallest generator of 256 is 18, which is 2 more
order of 256 is 21845, which is 3^th root of gen
smallest generator of 65536 is 258, which is 2 more
order of 65536 is 1431655765, which is 3^th root of gen
smallest generator of 4294967296 is 65540, which is 4 more


In [19]:
x = Nim(Ordinal(9)*2+Ordinal(2)+2)
y = Nim(7)
(y + x).__dict__

{'val': w**9*2 + w**2 + 5,
 'isfinite': False,
 'field': w**27,
 'base': w**9,
 'high': 2,
 'low': w**2 + 5}

In [7]:
# class Nim:
#     ''' finite nimbers '''
#     def __init__(self, n : int) -> None:
#         self.ord = abs(n)
#         # n = high * 2^(2 ^ (level-1)) + low;   high, low < 2^(2 ^ (level-1))
#         level = 0
#         if self.ord < 2:
#             self.level = level
#             self.fermat = None
#             self.high = 0
#             self.low = self.ord
#         else:
#             while n >> (1 << level) > 0:
#                 level += 1
#             self.level = level
#             self.fermat = (1 << (1 << (level - 1)))
#             self.high = self.ord // self.fermat
#             self.low = self.ord - self.high * self.fermat 
#     def __repr__(self) -> str:       
#         return str(self.ord)
    
#     def __add__(self, other):
#         return Nim(self.ord ^ other.ord)
    
#     def __mul__(self, other):
#         x = self.ord
#         y = other.ord
        
#         def nim_product(a : int, b : int) -> int:
#             # first handle trivial cases
#             if a == 0 or b == 0:
#                 return 0
#             elif a == 1:
#                 return b
#             elif b == 1:
#                 return a
#             elif a == 2 and b == 2:
#                 return 3
#             else:
#                 # do euclidean division by greatest possible fermat power 
#                 # a = q_a * F_a + r_a and b = q_b * F_b + r_b
#                 F_a, q_a, r_a = Nim(a).fermat, Nim(a).high, Nim(a).low
#                 F_b, q_b, r_b = Nim(b).fermat, Nim(b).high, Nim(b).low
                
#                 # if one the Fermat powers is greater than the other, then
#                 # nim multiplication by it is the same as ordinary multiplication
#                 if F_a < F_b:
#                     return nim_product(a,q_b)*F_b ^ nim_product(a,r_b)
#                 elif F_a > F_b:
#                     return nim_product(q_a,b)*F_a ^ nim_product(r_a,b)
#                 else:
#                     # otherwise we have to distribute and use F_n ** 2 = 3 * F_n / 2
#                     p_1 = nim_product(q_a,q_b)
#                     p_2 = nim_product(r_a,r_b)
#                     p_3 = nim_product(q_a ^ r_a, q_b ^ r_b)
#                     p_4 = nim_product(p_1, F_a >> 1)
#                     p_5 = p_3 ^ p_2
#                     return p_5 * F_a ^ p_2 ^ p_4
#         return Nim(nim_product(x, y))

In [8]:
def fermat_divisors(n : int, include_one : bool = False ) -> list:
                '''
                Find the divisors of a Mersenne number 2 ** (2 ** n) - 1
                By default does not include 1
                '''
                # for now, this only works for n < 6
                # could potentiall go up to n = 11 using known factors on wikipedia 
                # no one knows the factors of 2 ** (2 ** 11) + 1
                if n >= 6:
                    raise ValueError('This function only works for n < 6')
                else:
                    divisors = []
                    for i in range(0 + int(not include_one),2 ** n):
                        product = 1
                        for j in range(n):
                            if i >> j & 1:
                                product *= 2 ** (2 ** j) + 1
                        divisors.append(product)
                    return divisors